### Imprime exactamente 8 números:
* umbral (p95 o p90)
* n_total
* n_extreme_weeks
* n_episodes
* max_episode_weeks
* Δmean (outcome_anom, lag L)
* IC95 Δmean (bootstrap por semanas)
* IC95 Δmean (bootstrap por episodios)

Requiere columnas: week_start, tu variable x_var, tu outcome y_var.
Opcional: filtrar a 2018+ o lo que quieras.

## 1.Load Data

In [4]:
import pandas as pd
from pathlib import Path
import numpy as np

fp = Path(r"../../data/processed/gran_canaria/master/master_gcan_2015_2024.parquet")
df = pd.read_parquet(fp)

df["week_start"] = pd.to_datetime(df["week_start"], unit="ms", errors="coerce")
df = df.sort_values("week_start")

df[["week_start","deaths_week","PM10"]].head()

,week_start,deaths_week,PM10
0,2015-12-28,133.0,62.666667
1,2016-01-04,126.0,28.142857
2,2016-01-11,127.0,32.428571
3,2016-01-18,135.0,70.285714
4,2016-01-25,136.0,50.285714


In [6]:
def extreme_week_audit(
    df: pd.DataFrame,
    x_var: str,
    y_var: str = "deaths_week",
    date_col: str = "week_start",
    quantile: float = 0.95,
    lag: int = 2,
    seasonality: str = "woy",   # "woy" o "month"
    start_date: str | None = None,
    B: int = 3000,
    seed: int = 42,
):
    d = df[[date_col, x_var, y_var]].copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col, x_var, y_var]).sort_values(date_col)

    if start_date:
        d = d[d[date_col] >= pd.to_datetime(start_date)]

    n_total = len(d)
    thr = float(d[x_var].quantile(quantile))
    d["is_extreme"] = (d[x_var] > thr).astype(int)

    # episodes = consecutive extreme weeks
    start = (d["is_extreme"] == 1) & (d["is_extreme"].shift(1, fill_value=0) == 0)
    d["ep_id"] = start.cumsum()
    d.loc[d["is_extreme"] == 0, "ep_id"] = pd.NA

    eps = (
        d.dropna(subset=["ep_id"])
         .groupby("ep_id")
         .agg(weeks=(date_col, "size"))
    )
    n_episodes = int(len(eps))
    max_ep_weeks = int(eps["weeks"].max()) if n_episodes else 0
    n_ext_weeks = int(d["is_extreme"].sum())

    # lag on extreme flag (what happens to y at time t when extreme happened at t-lag)
    d["ext_lag"] = d["is_extreme"].shift(lag)
    d = d.dropna(subset=["ext_lag"])
    d["ext_lag"] = d["ext_lag"].astype(int)

    # seasonality adjustment for outcome: anomalies
    if seasonality == "woy":
        key = d[date_col].dt.isocalendar().week.astype(int)
    elif seasonality == "month":
        key = d[date_col].dt.month.astype(int)
    else:
        raise ValueError("seasonality must be 'woy' or 'month'")

    d["y_anom"] = d[y_var] - d.groupby(key)[y_var].transform("mean")

    g1 = d.loc[d["ext_lag"] == 1, "y_anom"].to_numpy()
    g0 = d.loc[d["ext_lag"] == 0, "y_anom"].to_numpy()

    d_mean = float(g1.mean() - g0.mean()) if (len(g1) and len(g0)) else np.nan

    # bootstrap by weeks
    rng = np.random.default_rng(seed)
    boot = np.empty(B)
    for b in range(B):
        s1 = rng.choice(g1, size=len(g1), replace=True)
        s0 = rng.choice(g0, size=len(g0), replace=True)
        boot[b] = s1.mean() - s0.mean()
    ci_weeks = np.quantile(boot, [0.025, 0.975])

    # bootstrap by episodes (more conservative)
    # sample episodes with replacement + keep all non-extreme-lag controls
    # We build a dataset from episodes, then recompute g1/g0 inside it (simple but robust)
    if n_episodes:
        # map ep_id for the original extreme weeks (not lagged); still a good block unit
        base0 = d[d["is_extreme"] == 0].copy()
        ep_ids = eps.index.to_numpy()

        boot_ep = []
        for _ in range(B):
            sample_ids = rng.choice(ep_ids, size=len(ep_ids), replace=True)
            sample1 = pd.concat([d[d["ep_id"] == sid] for sid in sample_ids], ignore_index=True)

            dd = pd.concat([base0, sample1], ignore_index=True).sort_values(date_col).copy()
            dd["ext_lag"] = dd["is_extreme"].shift(lag)
            dd = dd.dropna(subset=["ext_lag"])
            dd["ext_lag"] = dd["ext_lag"].astype(int)

            if seasonality == "woy":
                k2 = dd[date_col].dt.isocalendar().week.astype(int)
            else:
                k2 = dd[date_col].dt.month.astype(int)

            dd["y_anom"] = dd[y_var] - dd.groupby(k2)[y_var].transform("mean")
            gg1 = dd.loc[dd["ext_lag"] == 1, "y_anom"].to_numpy()
            gg0 = dd.loc[dd["ext_lag"] == 0, "y_anom"].to_numpy()

            if len(gg1) < 5 or len(gg0) < 20:
                continue
            boot_ep.append(float(gg1.mean() - gg0.mean()))

        boot_ep = np.array(boot_ep)
        ci_eps = np.quantile(boot_ep, [0.025, 0.975]) if len(boot_ep) else np.array([np.nan, np.nan])
    else:
        ci_eps = np.array([np.nan, np.nan])

    # ---- print exactly 8 numbers ----
    print(f"1) threshold_q{int(quantile*100)} = {thr:.4f}")
    print(f"2) n_total = {n_total}")
    print(f"3) n_extreme_weeks = {n_ext_weeks}")
    print(f"4) n_episodes = {n_episodes}")
    print(f"5) max_episode_weeks = {max_ep_weeks}")
    print(f"6) delta_mean_y_anom (lag{lag}) = {d_mean:.4f}")
    print(f"7) CI95 delta_mean (week-bootstrap) = [{ci_weeks[0]:.4f}, {ci_weeks[1]:.4f}]")
    print(f"8) CI95 delta_mean (episode-bootstrap) = [{ci_eps[0]:.4f}, {ci_eps[1]:.4f}]")

# --- Example call (GC PM10 extreme weeks, 2018+)
extreme_week_audit(df, x_var="PM10", y_var="deaths_week", quantile=0.90, lag=2, seasonality="woy", start_date="2018-01-01")

1) threshold_q90 = 85.7857
2) n_total = 366
3) n_extreme_weeks = 37
4) n_episodes = 23
5) max_episode_weeks = 5
6) delta_mean_y_anom (lag2) = 0.9412
7) CI95 delta_mean (week-bootstrap) = [-5.1494, 6.8660]
8) CI95 delta_mean (episode-bootstrap) = [-4.6418, 9.5346]
